In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch.ao.quantization import (
    get_default_qat_qconfig,
    QConfigMapping
)
from torch.ao.quantization.quantize_fx import (
    prepare_qat_fx,
    convert_fx
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
def replace_relu6(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU6):
            setattr(module, name, nn.ReLU(inplace=True))
        else:
            replace_relu6(child)

# UNCOMMENT ONLY if your model uses ReLU6 (MNV2/GNV2)
# replace_relu6(original_model)
# print("ReLU6 replaced with ReLU ✔")


In [ ]:
model_path = "/content/pruned_model_entire.pth"
original_model  = torch.load(model_path, map_location=device, weights_only=False)
original_model.eval()

print("original_model  loaded successfully!")

In [ ]:
total_params = sum(p.numel() for p in original_model.parameters())
print(f"Number of parameters in the original model: {total_params}")

In [ ]:
use RGB
train_transform = transforms.Compose([

    transforms.Resize (size = (224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  #^------------------ Normalize the images like ImageNet dataset ( these number are standard for ImageNet)
                         std=[0.229, 0.224, 0.225]),

])

val_transform = transforms.Compose([ #!No augmentation for validation !!!

  transforms.Resize (size = (224, 224)),

   transforms.ToTensor(),

      transforms.Normalize(mean=[0.485, 0.456, 0.406],  #^------------------ Normalize the images like ImageNet dataset ( these number are standard for ImageNet)
                         std=[0.229, 0.224, 0.225]),

 ])

train_dir = "colon_after_splitting/train"
val_dir   = "colon_after_splitting/val"


In [ ]:
# # -------------------------- USE_GRAYSCALE:
# train_transform = transforms.Compose([
#         transforms.Lambda(lambda img: img.convert("RGB")),
#         transforms.Resize((224, 224)),
#         transforms.RandomHorizontalFlip(0.5),
#         transforms.RandomRotation(10),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
# ])

# val_transform = transforms.Compose([
#         transforms.Lambda(lambda img: img.convert("RGB")),
#         transforms.Resize((224, 224)),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
# ])

# # train_dir = "COVID19+PNEUMONIA+NORMAL Chest X-Ray Image Dataset/train"
# # val_dir   = "COVID19+PNEUMONIA+NORMAL Chest X-Ray Image Dataset/val"


# train_dir = "K+F MRI (clean)/train"
# val_dir   = "K+F MRI (clean)/val"

In [ ]:

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

example_inputs, _ = next(iter(train_loader))
example_inputs = example_inputs.to("cpu")


In [ ]:
#fbgemm or qnnpack
backend = "qnnpack"
qconfig = get_default_qat_qconfig(backend)

qconfig_mapping = QConfigMapping().set_global(qconfig)
print("Model prepared for QAT ✔")


In [ ]:
original_model.to("cpu")
original_model.train()

prepared_qat_model = prepare_qat_fx(
    original_model,
    qconfig_mapping,
    example_inputs,
)
prepared_qat_model.to(device)
prepared_qat_model.train()

print("FX-QAT model prepared successfully!")

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

example_inputs, _ = next(iter(train_loader))
example_inputs = example_inputs.to("cpu")

In [ ]:
pos = 1  # colon = 0, xray/MRI = 1
criterion = nn.BCEWithLogitsLoss()
optimizer_qat = torch.optim.Adam(prepared_qat_model.parameters(), lr=1e-5)

num_epochs = 7 # adjust if needed

torch.manual_seed(42)

for epoch in range(num_epochs):
    prepared_qat_model.train()
    total_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)", ncols=80)

    for imgs, labels in train_bar:
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        optimizer_qat.zero_grad()
        logits = prepared_qat_model(imgs).squeeze()

        loss = criterion(logits, labels)
        loss.backward()
        optimizer_qat.step()

        total_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())
    prepared_qat_model.eval()

    all_preds = []
    all_labels = []

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Val)", ncols=80)

    with torch.no_grad():
        for imgs, labels in val_bar:
            imgs = imgs.to(device)
            labels = labels.float().to(device)

            logits = prepared_qat_model(imgs).squeeze()
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds, pos_label = pos)
    prec = precision_score(all_labels, all_preds, zero_division=0,pos_label = pos)
    rec  = recall_score(all_labels, all_preds, zero_division=0, pos_label = pos)
    f1   = f1_score(all_labels, all_preds, zero_division=0, pos_label = pos)

    print(f"\n Epoch {epoch+1}/{num_epochs}")
    print(f"Loss:      {total_loss/len(train_loader):.4f}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")


In [ ]:
prepared_qat_model.to("cpu")
prepared_qat_model.eval()

quantized_model = convert_fx(prepared_qat_model)
print("INT8 Model Ready")

In [ ]:
for name, m in quantized_model.named_modules():
    if hasattr(m, "weight"):
        try:
            w = m.weight()
            print(name, w.dtype)
        except:
            pass


In [ ]:
for name, module in quantized_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")


In [ ]:
for m in quantized_model.modules():
    if "Conv" in m.__class__.__name__:
        print(type(m))

# Saving

In [ ]:
# scripted_model = torch.jit.script(quantized_model)
# scripted_model.save("quantized_scripted.pt")


In [ ]:
# # Define the paths to save the model
# save_path_state_dict = "quantized_state.pth"
# save_path_entire_model = "quantized_entire.pth"

# torch.save(quantized_model.state_dict(), save_path_state_dict)
# print(f"Model state dictionary saved to {save_path_state_dict}")
# torch.save(quantized_model, save_path_entire_model)
# print(f"Entire model saved to {save_path_entire_model}")